In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
import xarray as xr

from climate_indices import compute, indices, spei, spi
from climate_indices.exceptions import CoordinateValidationError, InvalidArgumentError

In [ ]:
def _validate_monthly_time(time_values):
    """Return complete month-start or month-end timestamps."""
    try:
        time = pd.DatetimeIndex(time_values)
    except (TypeError, ValueError, OverflowError) as exc:
        raise CoordinateValidationError(
            "Time coordinate must contain supported datetime values.",
            coordinate_name="time",
            reason="not datetime-like",
        ) from exc
    if time.empty:
        raise CoordinateValidationError(
            "Time coordinate must not be empty.", coordinate_name="time", reason="empty coordinate"
        )
    if time.is_month_start.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq="MS")
    elif time.is_month_end.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq=pd.offsets.MonthEnd())
    else:
        expected_time = pd.DatetimeIndex([])
    if not time.equals(expected_time):
        raise CoordinateValidationError(
            "Time coordinate must be a complete, chronological sequence of monthly "
            "month-start or month-end timestamps.",
            coordinate_name="time",
            reason="non-monotonic, irregular, or gapped monthly timestamps",
        )
    return time

In [ ]:
from dask.distributed import Client

# Dashboard disabled so the optional bokeh dependency stays optional.
client = Client(n_workers=4, threads_per_worker=2, memory_limit="4GB", dashboard_address=None)

## Canonical calculation path

SPI and SPEI are computed through the public typed API — `climate_indices.spi` and
`climate_indices.spei` — on Dask-backed `xarray.DataArray`s. Under the hood the adapter
runs `xr.apply_ufunc(..., dask="parallelized")`, so labeled dimensions and coordinates
are preserved and results stay lazy until the deliberate Zarr write below
([ADR-0001](../docs/adr/0001-dual-numpy-xarray-api.md),
[ADR-0002](../docs/adr/0002-multiprocessing-cli-dask-xarray.md)).

Dask parallelizes across spatial chunks: each `lat`/`lon` block of the prepared store is
an independent task, while `time` stays a single chunk so distribution fitting sees the
full series at each location
([ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md)). Worker count alone
is not evidence of parallelism — inspect the chunk layout and task graph instead.
Precipitation and PET were aligned with `join="exact"` at preparation time, so SPEI
never relies on xarray's coordinate intersection.

In [ ]:
# Canonical configuration: matches data/e2e/manifest.json (1980-2016 monthly
# inputs, 1981-2010 Calibration Period).
pipeline_config = {
    "scale": 3,  # 3-Month SPI/SPEI
    "distribution_spi": indices.Distribution.gamma,
    "distribution_spei": indices.Distribution.pearson,
    "data_start_year": 1980,
    "cal_start_year": 1981,
    "cal_end_year": 2010,
    "periodicity": compute.Periodicity.monthly,
}

# prepare_e2e_inputs.py atomically publishes this validated input generation.
# CLIMATE_INDICES_E2E_DATA overrides the default location (used by the test suite).
data_root = Path(os.environ.get("CLIMATE_INDICES_E2E_DATA", "../data/e2e"))
if not (data_root / "current").exists():
    raise FileNotFoundError(
        f"Prepared inputs not found: {data_root / 'current'}. "
        "Generate them once with: uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py"
    )
data_dir = (data_root / "current").resolve()
prepared_zarr = data_dir / "cache_prepared_input.zarr"
final_output_zarr = data_root / "climate_indices_output.zarr"

In [ ]:
ds = xr.open_zarr(prepared_zarr, consolidated=True).transpose("time", "lat", "lon").chunk({"time": -1})

time = _validate_monthly_time(ds["time"].values)
if time[0].month != 1 or time[-1].month != 12:
    raise CoordinateValidationError(
        "Prepared dataset must cover complete calendar years.",
        coordinate_name="time",
        reason="incomplete first or final year",
    )
data_start_year = pipeline_config["data_start_year"]
if time[0].year != data_start_year:
    raise InvalidArgumentError(
        "pipeline_config['data_start_year'] does not match the prepared dataset's first monthly timestamp.",
        argument_name="data_start_year",
        argument_value=str(data_start_year),
        valid_values=str(time[0].year),
    )
cal_start_year, cal_end_year = pipeline_config["cal_start_year"], pipeline_config["cal_end_year"]
if not (data_start_year <= cal_start_year <= cal_end_year <= time[-1].year):
    raise InvalidArgumentError(
        "Calibration Period must fall within the prepared dataset's covered years.",
        argument_name="cal_start_year/cal_end_year",
        argument_value=f"{cal_start_year}-{cal_end_year}",
        valid_values=f"{data_start_year}-{time[-1].year}",
    )

In [ ]:
index_kwargs = {
    "scale": pipeline_config["scale"],
    "data_start_year": data_start_year,
    "calibration_year_initial": cal_start_year,
    "calibration_year_final": cal_end_year,
    "periodicity": pipeline_config["periodicity"],
}
spi_da = spi(values=ds["precip"], distribution=pipeline_config["distribution_spi"], **index_kwargs)
spei_da = spei(
    precips_mm=ds["precip"], pet_mm=ds["pet"], distribution=pipeline_config["distribution_spei"], **index_kwargs
)
spi_name, spei_name = f"spi_{pipeline_config['scale']}", f"spei_{pipeline_config['scale']}"
ds_output = xr.Dataset({spi_name: spi_da, spei_name: spei_da}, coords=ds.coords)

In [ ]:
# Write beside the target and swap on success so a failed run leaves a
# previously completed store intact.
tmp_path = final_output_zarr.with_name(final_output_zarr.name + ".tmp")
shutil.rmtree(tmp_path, ignore_errors=True)
# The typed API computes in float64 (xr.apply_ufunc(..., output_dtypes=[float])
# regardless of input dtype); downcast on write only, to keep the on-disk
# footprint at the float32 precision the float32 mm inputs actually carry.
float32_encoding = {"dtype": "float32"}
ds_output.to_zarr(
    tmp_path,
    mode="w",
    zarr_format=2,
    consolidated=True,
    encoding={spi_name: float32_encoding, spei_name: float32_encoding},
)
shutil.rmtree(final_output_zarr, ignore_errors=True)
tmp_path.rename(final_output_zarr)

In [ ]:
out_path = Path(final_output_zarr)

In [ ]:
out_path

In [ ]:
out_ds = xr.load_dataset(out_path)

In [ ]:
out_ds

In [ ]:
spi_index = out_ds['spi_3']
spei_index = out_ds['spei_3']

In [ ]:
spi_index.isel(time=9).plot(
    levels=8,
)

In [ ]:
spei_index.isel(time=9).plot(
    levels=8,
)